# Batched LoRA Adapter Evaluation

We will implement the evaluation pipeline for a LoRA fine tuned intent classifier. After fine tuning a base model (covered in Module 06), we need to measure how well the adapter actually performs on held out test data and track the results in MLflow.

## What we will do
- How to load a LoRA adapter on top of a quantized base model for inference
- How to clean and parse structured JSON output from a generative model
- How to translate enterprise data schemas into model specific chat templates
- How left padding enables correct batched generation
- How to run batched inference to speed up evaluation on a GPU
- How to compute accuracy and custom metrics (invalid JSON rate)
- How MLflow evaluate auto generates classification reports, confusion matrices and per class metrics
- How quality gates (MetricThreshold) can fail a pipeline run if accuracy drops below a threshold

In [1]:
!pip install transformers==4.57.1 datasets==4.0.0 peft==0.17.0 trl==0.21.0 accelerate==1.10.0 bitsandbytes==0.48.1
import os

!pip install "git+https://github.com/red-hat-data-services/mlflow@rhoai-3.3"


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
  Cloning https://github.com/red-hat-data-services/mlflow (to revision rhoai-3.3) to /tmp/pip-req-build-r3bshchc
  Running command git clone --filter=blob:none --quiet https://github.com/red-hat-data-services/mlflow /tmp/pip-req-build-r3bshchc
  Running command git checkout -b rhoai-3.3 --track origin/rhoai-3.3
  Switched to a new branch 'rhoai-3.3'
  branch 'rhoai-3.3' set up to track 'origin/rhoai-3.3'.
  Resolved https://github.com/red-hat-data-services/mlflow to commit c0f86528f988088bc07be8acc0db50074acee0f4
  Running command git submodule update --init --recursive -q
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


---
## 1 Imports and Dependencies

The evaluation script uses a mix of Hugging Face libraries for model loading and inference, MLflow for experiment tracking and sklearn for accuracy computation.

| Library | Purpose |
|---|---|
| `transformers` | Load the base model (`AutoModelForCausalLM`) and tokenizer, configure 4 bit quantization |
| `peft` | Load the LoRA adapter on top of the base model (`PeftModel`) |
| `mlflow` | Log evaluation metrics, datasets, failure artifacts and run quality gates |
| `sklearn` | `accuracy_score` for computing overall classification accuracy |
| `torch` | Tensor operations and disabling gradient computation during inference |
| `datasets` | Hugging Face dataset loading utilities |
| `tqdm` | Progress bar for batch iteration |
| `pandas` | Build DataFrames for MLflow evaluation and failure logging |

In [2]:
import mlflow
import json
import torch
import os
import re
import pandas as pd
from mlflow.models import make_metric, MetricThreshold
from mlflow.metrics.base import MetricValue
from sklearn.metrics import accuracy_score
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from tqdm import tqdm
from datasets import load_dataset

/opt/app-root/lib64/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


---
## 2 Cleaning JSON Output from the Model

Generative models do not always produce clean JSON. Depending on the base model and how it was fine tuned, the output might include:

- `<think>...</think>` reasoning blocks (common with DeepSeek and similar models)
- Markdown code fences like ` ```json ``` `
- Trailing text or garbage after the closing brace
- Truncated JSON if `max_new_tokens` cuts the output short

The `clean_json_output` function strips all of this away and tries to extract a valid JSON object. If the JSON is truncated (more opening than closing braces), it attempts a repair by appending the missing closing braces. This is a pragmatic approach that works well in practice for short JSON outputs like `{"intent": "some_label"}`.

In [3]:
def clean_json_output(text): 
    """
    extract valid JSON from model output text.
    handles:
    - <think>...</think> blocks
    - markdown code fences like ```json``` or ```
    - trailing/leading garbage
    - truncated JSON (attempts to recover)
    returns a dict if successful, else None
    """
    # aggressive removal of <think> blocks (even unclosed ones)
    text = re.sub(r'<think>.*?(?:</think>|$)', '', text, flags=re.DOTALL)

    # remove ```json``` or ``` fences
    text = re.sub(r'```json', '', text, flags=re.IGNORECASE)
    text = re.sub(r'```', '', text)

    # keep only the first { ... } block
    match = re.search(r'\{.*\}', text, flags=re.DOTALL)
    if match:
        json_str = match.group(0)
    else:
        # fallback: try entire text
        json_str = text.strip()

    # try to parse
    try:
        return json.loads(json_str)
    except json.JSONDecodeError:
        # attempt to fix truncated JSON by adding closing }
        if json_str.count('{') > json_str.count('}'):
            json_str += '}' * (json_str.count('{') - json_str.count('}'))
        try:
            return json.loads(json_str)
        except:
            return None

Let's test the cleaner with a few examples to see it in action.

In [4]:
# test with a clean json output
print(clean_json_output('{"intent": "check_balance"}'))

# test with think block wrapping the json
print(clean_json_output('<think>reasoning here</think>{"intent": "pay_bill"}'))

# test with markdown code fences
print(clean_json_output('```json\n{"intent": "transfer_money"}\n```'))

# test with truncated json
print(clean_json_output('{"intent": "open_account"'))

{'intent': 'check_balance'}
{'intent': 'pay_bill'}
{'intent': 'transfer_money'}
{'intent': 'open_account'}


---
## 3 Formatting Messages for Different Model Families

Different model families expect slightly different chat template structures. The key difference is how the system prompt is handled:

- Qwen, Phi and DeepSeek all support a `system` role in the message list
- Gemma does not support a `system` role, so the system prompt is prepended to the first user message instead

The function also translates the enterprise data schema (where bots are `"role": "assistant"`) into the Hugging Face convention (`"role": "assistant"`) and appends the current user message at the end.

In [5]:
def format_messages_for_model(row: dict, base_model_id: str) -> list:
    """
    translates the enterprise data schema into a model-specific chat template format
    """
    messages = []
    system_prompt = "You are an AI intent classifier. Analyze the conversation history and the latest user message. You must output only a valid JSON object containing the predicted 'intent'."
    
    model_name = base_model_id.lower()
    
    if "qwen" in model_name or "phi" in model_name or "deepseek" in model_name:
        messages.append({"role": "system", "content": system_prompt})
    elif "gemma" in model_name:
        pass 
    else:
        messages.append({"role": "system", "content": system_prompt})

    for turn in row.get("session_history", []):
        hf_role = "assistant" if turn.get("role") == "assistant" else "user"
        messages.append({"role": hf_role, "content": turn.get("content", "")})
        
    current_msg = row.get("user_message", "")
    
    if "gemma" in model_name and len(messages) == 0:
        current_msg = f"{system_prompt}\n\n{current_msg}"
        
    messages.append({"role": "user", "content": current_msg})
    
    return messages

Let's verify the formatting for both a system role model (Qwen) and a non system role model (Gemma).

In [6]:
sample_row = {
    "session_history": [
        {"role": "user", "content": "Hi, I need help"},
        {"role": "assistant", "content": "Sure, how can I help you?"},
    ],
    "user_message": "I want to check my balance",
    "intent": "check_balance"
}

print("=== Qwen format ===")
for msg in format_messages_for_model(sample_row, "Qwen/Qwen2.5-3B-Instruct"):
    print(f"  {msg['role']}: {msg['content'][:60]}")

print("\n=== Gemma format ===")
for msg in format_messages_for_model(sample_row, "google/gemma-2-9b-it"):
    print(f"  {msg['role']}: {msg['content'][:80]}")

=== Qwen format ===
  system: You are an AI intent classifier. Analyze the conversation hi
  user: Hi, I need help
  assistant: Sure, how can I help you?
  user: I want to check my balance

=== Gemma format ===
  user: Hi, I need help
  assistant: Sure, how can I help you?
  user: I want to check my balance


---
## 4 Model Loading with QLoRA and Adapter Merging

The evaluation function starts by loading the base model in 4 bit quantized mode using the same `BitsAndBytesConfig` used during training. This keeps GPU memory usage low enough to fit on a single T4 GPU (16 GB VRAM) even when running batched inference.

Key decisions here:
- `nf4` quantization type gives better accuracy than fp4 for language models
- Double quantization (`bnb_4bit_use_double_quant=True`) quantizes the quantization constants too, saving additional memory
- `float16` compute dtype keeps matrix multiplications fast on consumer GPUs
- The LoRA adapter is loaded on top of the quantized base model using `PeftModel.from_pretrained`
- `model.eval()` disables dropout layers so inference is deterministic

In [11]:
# configure these paths for your environment
base_model_id = os.environ.get("BASE_MODEL_ID", "Qwen/Qwen2.5-3B-Instruct")
adapter_path = os.environ.get("ADAPTER_PATH", "./mnt/data/adapters/checkpoint-120")
test_data_path = os.environ.get("TEST_SPLIT_PATH", "../mnt/data/test_split.jsonl")
dataset_source = os.environ.get("DATASET_SOURCE", "") #s3://path-to-actual-dataset-json-file
batch_size = 16

if not adapter_path:
    raise RuntimeError(
        "environment variable ADAPTER_PATH is missing or empty. "
        "please set ADAPTER_PATH to the LoRA adapter location."
    )

In [8]:
# 4 bit quantization keeps the 3B model under ~3 GB VRAM
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

print(f"loading base model {base_model_id}...")
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id, 
    device_map="auto", 
    quantization_config=bnb_config,
    trust_remote_code=True
)

print(f"loading LoRA adapter from {adapter_path}...")
model = PeftModel.from_pretrained(base_model, adapter_path)
model.eval()

loading base model Qwen/Qwen2.5-3B-Instruct...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]/opt/app-root/lib64/python3.11/site-packages/bitsandbytes/backends/cuda/ops.py:212: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading checkpoint shards: 100%|██████████| 2/2 [00:46<00:00, 23.04s/it]


loading LoRA adapter from ./mnt/data/adapters/checkpoint-120...


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 2048)
        (layers): ModuleList(
          (0-35): 36 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2048, out_features=2048, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora

---
## 5 Tokenizer Configuration: Why Left Padding Matters

When generating text one sample at a time, padding does not matter. But for batched generation, sequences of different lengths need to be padded to the same length before being stacked into a tensor.

Right padding (the default) puts pad tokens at the end. This breaks autoregressive generation because the model sees pad tokens in the middle of what it thinks is the sequence. Left padding puts pad tokens at the beginning so that every sequence ends at the real content boundary, and generation can continue naturally from there.

This is one of the most common gotchas when moving from single sample to batched generation.

In [9]:
tokenizer = AutoTokenizer.from_pretrained(base_model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
# critical: left padding is required for batched generation
tokenizer.padding_side = "left"

print(f"pad token: {tokenizer.pad_token}")
print(f"padding side: {tokenizer.padding_side}")

pad token: <|im_end|>
padding side: left


---
## 6 Loading the Test Dataset

The test data is a JSONL file where each line is a JSON object representing one evaluation sample. Each sample contains:
- `session_history`: a list of prior conversation turns
- `message`: the current user message to classify
- `intent`: the ground truth label

In [12]:
with open(test_data_path) as f:
    data = [json.loads(line) for line in f]

print(f"loaded {len(data)} test samples")
print(f"sample keys: {list(data[0].keys())}")
print(f"sample intent: {data[0].get('intent')}")

loaded 47 test samples
sample keys: ['user_message', 'intent', 'session_history']
sample intent: MOBILE_BILLING_CHECK_DUE_DATE


---
## 7 Batched Inference Loop

This is the core of the evaluation pipeline. Instead of running inference one sample at a time (which underutilizes the GPU), we process samples in batches. The steps for each batch are:

1. Format all samples into chat template messages using `format_messages_for_model`
2. Apply the tokenizer's chat template to produce raw prompt strings (with special tokens, turn markers etc.)
3. Tokenize the batch with left padding so all sequences have the same length
4. Run `model.generate` with `torch.no_grad()` to produce output tokens
5. Slice off the prompt tokens to isolate only the newly generated tokens
6. Decode the generated tokens and parse the JSON output

Generation parameters:
- `max_new_tokens=50`: hard cap since intent JSON output is always short
- `temperature=0.0` and `do_sample=False`: greedy decoding for deterministic results
- `eos_token_id` and `pad_token_id`: prevent the model from generating beyond the end of the output

Failures are categorized into:
- `INVALID_JSON`: the model produced something that could not be parsed as JSON
- `WRONG_INTENT`: valid JSON but the predicted intent does not match ground truth
- `EXCEPTION`: unexpected error during parsing

In [13]:
failures = [] 
y_true = []
y_pred = []

print(f"running batched evaluation (batch_size={batch_size})...")

for i in tqdm(range(0, len(data), batch_size)):
    batch = data[i : i + batch_size]
    
    batch_messages = [format_messages_for_model(row, base_model_id) for row in batch]
    batch_ground_truths = [row.get("intent", "UNKNOWN") for row in batch]
    
    # step 1: apply chat template to get raw text strings  
    prompt_texts = []
    for msgs in batch_messages:
        raw_prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        raw_prompt += '{"intent": "'
        prompt_texts.append(raw_prompt)
    
    # step 2: tokenize the batch with left padding
    inputs = tokenizer(
        prompt_texts, 
        padding=True, 
        return_tensors="pt"
    ).to(base_model.device)

    # step 3: generate predictions with greedy decoding
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=50,
            temperature=0.0,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
        )

    # step 4: slice off the prompt to only decode newly generated tokens
    prompt_length = inputs["input_ids"].shape[1]
    generated_tokens = outputs[:, prompt_length:]
    pred_texts = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)

    # step 5: parse outputs and log failures
    for row, ground_truth, pred_text in zip(batch, batch_ground_truths, pred_texts):
        pred_intent = "INVALID_JSON"
        full_output_string = '{"intent": "' + pred_text
        error_type = None

        try:
            pred_json = clean_json_output(full_output_string)
            if pred_json:
                pred_intent = pred_json.get("intent", "UNKNOWN")
                if pred_intent != ground_truth:
                    error_type = "WRONG_INTENT"
            else:
                pred_intent = "INVALID_JSON"
                error_type = "INVALID_JSON"
        except Exception as e:
            error_type = f"EXCEPTION: {str(e)}"

        if error_type:
            failures.append({
                "type": error_type,
                "input": row.get("user_message", ""),
                "ground_truth": ground_truth,
                "prediction": pred_intent,
                "raw_output": pred_text
            })                

        y_true.append(ground_truth)
        y_pred.append(pred_intent)

running batched evaluation (batch_size=16)...


100%|██████████| 3/3 [00:12<00:00,  4.31s/it]


---
## 8 Quick Accuracy Check

Before logging to MLflow, a quick sanity check on the raw accuracy. This gives immediate feedback on how the adapter is performing.

In [14]:
accuracy = accuracy_score(y_true, y_pred)
print(f"final accuracy: {accuracy * 100:.2f}%")
print(f"total samples: {len(y_true)}")
print(f"correct: {sum(t == p for t, p in zip(y_true, y_pred))}")
print(f"failures logged: {len(failures)}")

final accuracy: 95.74%
total samples: 47
correct: 45
failures logged: 2


---
## 9 Custom Metric: Invalid JSON Rate

Accuracy alone does not tell the full story. A model might produce the right intent sometimes but fail to produce valid JSON at all in other cases. The invalid JSON rate tracks this separately.

This is important because:
- A wrong intent means the model understood the format but misclassified. This can improve with more training data.
- An invalid JSON means the model has a structural output problem. This might indicate the adapter has regressed on instruction following.

MLflow's `make_metric` API lets us define custom evaluation metrics that plug into the same evaluation framework as built in metrics.

In [15]:
# tracks how often the model fails to produce parseable JSON
def invalid_json_rate_fn(predictions, targets, metrics):
    rate = sum(p == "INVALID_JSON" for p in predictions) / len(predictions)
    return MetricValue(aggregate_results={"invalid_json_rate": rate})

invalid_json_metric = make_metric(
    eval_fn=invalid_json_rate_fn,
    greater_is_better=False,
    name="invalid_json"
)

# preview the rate before MLflow logging
invalid_count = sum(p == "INVALID_JSON" for p in y_pred)
print(f"invalid JSON predictions: {invalid_count}/{len(y_pred)} ({invalid_count/len(y_pred)*100:.2f}%)")

invalid JSON predictions: 0/47 (0.00%)


---
## 10 MLflow Evaluation and Quality Gate

This is where everything comes together. The MLflow evaluation step does several things automatically:

1. Computes standard classification metrics: accuracy, precision, recall and F1 per class
2. Generates a confusion matrix as a PNG artifact
3. Produces a full classification report
4. Evaluates our custom `invalid_json` metric alongside the built in ones

After evaluation, the quality gate (`mlflow.validate_evaluation_results`) checks if accuracy meets a minimum threshold (50% in this case). If it does not, MLflow raises a `ModelValidationFailedException`. In a CI/CD context (like an OpenShift training job), this causes the eval pod to exit non zero, failing the pipeline and preventing a bad adapter from being promoted.

The failures list is also logged as both a JSON artifact and a table artifact, making it easy to inspect specific misclassifications in the MLflow UI.

In [16]:
import os

os.environ["MLFLOW_TRACKING_TOKEN"] = "sha256~UQp-jGiZwIBucKRnhpTZEH3lMX8cndC8likfV78vo-g" # your token here
os.environ["MLFLOW_TRACKING_URI"] = "https://mlflow.redhat-ods-applications.svc.cluster.local:8443"
os.environ["MLFLOW_WORKSPACE"] = "decoder-sft"
os.environ["MLFLOW_RUN_NAME"] = "decoder-eval-run"
os.environ["MLFLOW_TRACKING_HEADERS"] = "{\"X-MLFLOW-WORKSPACE\": \"decoder-sft\", \"Content-Type\": \"application/json\"}"
os.environ["MLFLOW_EXPERIMENT_NAME"] = "decoder-evaluation"
os.environ["MLFLOW_TRACKING_INSECURE_TLS"] = "true"
print("Environment variables set!")

Environment variables set!


In [17]:
eval_df = pd.DataFrame({"prediction": y_pred, "target": y_true})

mlflow_run_name = os.environ.get("MLFLOW_RUN_NAME", "eval_run_v1")
with mlflow.start_run(run_name=mlflow_run_name):
    # log dataset lineage so we can trace which data produced these results
    if dataset_source:
        mlflow_dataset_eval = mlflow.data.from_pandas(
            pd.DataFrame(data),
            source=dataset_source,
            name="intent_eval_data"
        )
        mlflow.log_input(mlflow_dataset_eval, context="evaluation")

    # auto generates accuracy, precision/recall/f1 per class,
    # confusion matrix png and classification report
    result = mlflow.models.evaluate(
        data=eval_df,
        predictions="prediction",
        targets="target",
        model_type="classifier",
        extra_metrics=[invalid_json_metric],
    )

    # quality gate: raises ModelValidationFailedException if accuracy
    # drops below threshold, failing the eval job in the pipeline
    mlflow.validate_evaluation_results(
        candidate_result=result,
        validation_thresholds={
            "accuracy_score": MetricThreshold(threshold=0.50, greater_is_better=True),
        }
    )

    # log failure details for debugging in the MLflow UI
    if failures:
        mlflow.log_dict(failures, "failures.json")
        error_df = pd.DataFrame(failures)
        mlflow.log_table(data=error_df, artifact_file="misclassifications.json")

print("evaluation complete. check the MLflow UI for full results.")

/opt/app-root/lib64/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'mlflow.redhat-ods-applications.svc.cluster.local'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/opt/app-root/lib64/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'mlflow.redhat-ods-applications.svc.cluster.local'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/opt/app-root/lib64/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'mlflow.redhat-ods-applications.svc.cluster.local'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/adv

🏃 View run decoder-eval-run at: https://mlflow.redhat-ods-applications.svc.cluster.local:8443/#/workspaces/decoder-sft/experiments/9/runs/67b3ba6b21204366891893c3bf83c822
🧪 View experiment at: https://mlflow.redhat-ods-applications.svc.cluster.local:8443/#/workspaces/decoder-sft/experiments/9
evaluation complete. check the MLflow UI for full results.


/opt/app-root/lib64/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'mlflow.redhat-ods-applications.svc.cluster.local'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/opt/app-root/lib64/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'mlflow.redhat-ods-applications.svc.cluster.local'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/opt/app-root/lib64/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'mlflow.redhat-ods-applications.svc.cluster.local'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/adv

<Figure size 1050x700 with 0 Axes>

---
## 11 Inspecting Failures

If the model produced any failures, this section gives a quick look at what went wrong. This is helpful for spotting patterns like:
- Are certain intents consistently misclassified?
- Is the model producing truncated output for longer conversation histories?
- Are invalid JSON outputs concentrated on specific types of inputs?

In [18]:
if failures:
    error_df = pd.DataFrame(failures)
    print(f"\nfailure breakdown by type:")
    print(error_df["type"].value_counts().to_string())
    print(f"\nfirst 5 failures:")
    for f in failures[:5]:
        print(f"  type={f['type']} | expected={f['ground_truth']} | got={f['prediction']}")
        print(f"  raw: {f['raw_output'][:100]}")
        print()
else:
    print("no failures! all predictions matched ground truth.")


failure breakdown by type:
type
WRONG_INTENT    2

first 5 failures:
  type=WRONG_INTENT | expected=MOBILE_USAGE_CHECK_DATA_CURRENT | got=MOBILE_BILLING_CHECK_DATA_CURRENT
  raw: MOBILE_BILLING_CHECK_DATA_CURRENT"}

  type=WRONG_INTENT | expected=MOBILE_NETWORK_CHECK_NO_SIGNAL | got=MOBILE_BILLING_PAYMENT_STATUS_NO_SIGNAL
  raw: MOBILE_BILLING_PAYMENT_STATUS_NO_SIGNAL"}



---
## Summary

We covered the complete evaluation pipeline for a LoRA fine tuned intent classifier:

| Step | What it does |
|---|---|
| JSON cleaning | Strips think blocks, code fences and garbage from model output |
| Message formatting | Translates enterprise data schema to model specific chat templates |
| Model loading | Loads 4 bit quantized base model + LoRA adapter |
| Left padding | Enables correct batched generation for autoregressive models |
| Batched inference | Processes multiple samples per GPU pass for faster evaluation |
| MLflow evaluate | Auto generates accuracy, per class metrics and confusion matrix |
| Quality gate | Fails the pipeline if accuracy drops below 50% |
| Failure logging | Saves misclassifications as artifacts for debugging |

The next step after evaluation is typically deploying the adapter using a serving framework like vLLM.